In [15]:
import os
os.environ["DDE_BACKEND"] = "pytorch"

In [1]:
import deepxde as dde
dde.backend.backend_name = "pytorch"

import numpy as np
import matplotlib.pyplot as plt
import torch

2026-08-03 21:58:58.500334: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-03 21:58:58.706380: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-03 21:58:58.706417: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-03 21:58:58.744907: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-03 21:58:58.826039: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-03 21:58:58.827085: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [4]:
def pde(x, y):
    dy_dt = dde.grad.jacobian(y, x, i=0, j=1)
    dy_dx = dde.grad.jacobian(y, x, i=0, j=0)
    dy_xx = dde.grad.hessian(y, x, component=0, i=0, j=0)
    return dy_dt + y*dy_dx - 0.01 / np.pi * dy_xx


In [8]:
#Spatial Boundary
def boundary(X, on_boundary):

    if not on_boundary:
        return False

    return np.isclose(X[0], -1) or np.isclose(X[0], 1)

def boundary_value(X):
    return 0

#Initial condition
def initial_condition(X, on_boundary):
    if not on_boundary:
        return False

    return np.isclose(X[1], 0)

def initial_condition_value(X):
    return - np.sin(np.pi*X[:, 0:1])

In [7]:
#defining the spatial and time domains and combining them for the final interval
geom = dde.geometry.Interval(-1, 1)
timedomain = dde.geometry.TimeDomain(0, 1.0)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [10]:
bc = dde.icbc.DirichletBC(geomtime, boundary_value, boundary)
ic = dde.icbc.IC(geomtime, initial_condition_value, initial_condition)

In [11]:
data = dde.data.TimePDE(geomtime, pde, [bc, ic], num_domain=2540, num_boundary=80, num_initial=160)

In [ ]:
#model definition
layer_size = [2] + [20]*3 + [1] #2 input variables(x, t)
activation = "tanh"
initializer = "Glorot uniform"

net = dde.nn.FNN(layer_size, activation, initializer)

model = dde.Model(data, net)

In [ ]:
#Custom metric function which can be replaced by an external dataset. X_ref and Y_ref are from the dataset
def l2_relative_error_external(_, y_pred):
    y_test_pred = model.predict(X_ref)
    return dde.metrics.l2_relative_error(Y_ref, y_test_pred)

In [ ]:
model.compile("adam", lr=0.001, metrics=[l2_relative_error_external])

losshistory, train_state = model.train(iterations=15000)

Compiling model...
'compile' took 0.004960 s

Training model...



/media/srimukha-sarma/Windows-SSD/conda-envs/deepxde/lib/python3.11/site-packages/keras/src/initializers/initializers.py:120: UserWarning: The initializer GlorotUniform is unseeded and being called multiple times, which will return identical values each time (even if the initializer is unseeded). Please update your code to provide a seed to the initializer, or avoid using the same initializer instance more than once.
  warnings.warn(


NameError: name 'X_ref' is not defined

In [ ]:
#Saving the model
model.save(save_path="/models/best.h", protocol="tensorflow")